# 📖 Notebook 1: Bid Processing & Concurrency

Bidding is the heart of an online auction. When two people bid at the same time, bad things can happen — both bids get accepted, the wrong person wins, or bids silently disappear.

In this notebook, we'll **break the system on purpose** to see these race conditions, then fix them with three different approaches.

## Learning Objectives

By the end of this notebook, you'll understand:
- Why naive read-then-write creates race conditions
- How **row-level locking** (pessimistic) serializes bid processing
- How **optimistic concurrency control** (OCC) avoids locks entirely
- How **Redis Lua scripts** provide atomic compare-and-set operations
- The tradeoffs between each approach

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/online-auction
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `auction_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import time
import threading

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "auction_demo",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    """Create a new PostgreSQL connection."""
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    """Create a new Redis client."""
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 📋 Requirements & Back-of-the-Envelope

### Functional
1. A seller **lists an item** with a starting price and an end date.
2. A bidder **places a bid**; it is accepted only if it beats the current highest.
3. Anyone can **view an auction** and see the current highest bid.
4. When the end date passes, the auction **closes** and the highest bidder wins.

**Out of scope**: payments, shipping, seller reputation, search.

### Non-functional
| Requirement | Target | Why |
|---|---|---|
| Bid consistency | **Strong** | Every viewer must see the same highest bid, and it must be the real one |
| Durability | No accepted bid is ever lost | The bid table is legal evidence in a dispute |
| Bid latency | p99 < 500 ms | Bidders in the last seconds need immediate feedback |
| Read latency | p99 < 200 ms | Watchers refresh constantly near the end |
| Availability | 99.99% during the closing window | An outage at the close of a $50k auction is a lawsuit, not an incident |

This is the rare design where **consistency beats availability** and you should
say so explicitly. Showing a stale bid is not a degraded experience, it is a
wrong answer — so if the write path can't be made consistent, refuse the bid.

In [ ]:
# ── Back-of-the-envelope ────────────────────────────────────────────────
CONCURRENT_AUCTIONS  = 10_000_000
AUCTION_DAYS         = 7
BIDS_PER_AUCTION     = 100
END_GAME_MULTIPLIER  = 10      # bids pile up in the final minute (sniping)
VIEWS_PER_BID        = 100
BID_ROW_BYTES        = 200     # row + the two indexes on it
SEC_PER_DAY          = 86_400

auctions_per_day = CONCURRENT_AUCTIONS / AUCTION_DAYS
bids_per_day     = auctions_per_day * BIDS_PER_AUCTION
bid_rate         = bids_per_day / SEC_PER_DAY
peak_bid_rate    = bid_rate * END_GAME_MULTIPLIER

read_rate = bid_rate * VIEWS_PER_BID

storage_year = bids_per_day * 365 * BID_ROW_BYTES

# Contention is what actually limits us: how many bids hit ONE auction row?
HOT_AUCTION_BIDS_PER_S = 50        # a celebrity item in its final seconds
LOCK_HOLD_MS = 2                   # a tight, indexed bid transaction

print("📐 Back-of-the-envelope")
print("=" * 70)
print(f"  Concurrent auctions:   {CONCURRENT_AUCTIONS:>14,}")
print(f"  New auctions/day:      {auctions_per_day:>14,.0f}")
print(f"  Bids:                  {bid_rate:>14,.0f} /s avg   "
      f"{peak_bid_rate:>10,.0f} /s peak")
print(f"  Auction page reads:    {read_rate:>14,.0f} /s")
print(f"  Read : write ratio     {VIEWS_PER_BID:>14,} : 1")
print()
print(f"  Bid storage:           {storage_year / 1e12:>14,.1f} TB/year "
      f"({bids_per_day * BID_ROW_BYTES / 1e9:,.0f} GB/day)")
print()
print("  Per-auction contention (the number that actually matters):")
print(f"    Hot auction:         {HOT_AUCTION_BIDS_PER_S:>14,} bids/s on ONE row")
print(f"    Lock hold time:      {LOCK_HOLD_MS:>14,} ms")
print(f"    Max serial bids/s:   {1000 / LOCK_HOLD_MS:>14,.0f} on that row")
util = HOT_AUCTION_BIDS_PER_S / (1000 / LOCK_HOLD_MS)
print(f"    Lock utilisation:    {util:>13.0%}  "
      f"{'✅ fine' if util < 0.5 else '⚠️  queueing'}")

### What those numbers decide

- **~1,650 bids/s average, ~16,500/s at peak.** Bids cluster violently at the
  end of an auction — sniping is not an edge case, it is the normal shape of the
  traffic. Size for the last minute, not the average.
- **~165,000 auction-page reads/s against ~1,650 writes/s — 100:1.** So the
  *read* path gets a cache, and the *write* path gets a lock. Different
  consistency requirements, different mechanisms, same row.
- **~10 TB/year of bid history**, and you may not prune it. Bid history is the
  audit trail for "I bid $500, why didn't I win?". Partition by month, move old
  partitions to cheap storage, delete nothing.
- **The scale number that matters is not global throughput — it's contention on
  one row.** 16,500 bids/s spread over 10 million auctions is nothing. 50 bids/s
  on a *single* auction row in its final seconds is the whole problem, because
  those 50 must serialise. At a 2 ms lock hold that's 10% utilisation and fine;
  at a 200 ms hold (say, a network call inside the transaction) it's 1000% and
  the auction falls over at exactly the worst moment.

**Which is the real lesson of this notebook:** keep the bid transaction tiny.
No HTTP calls, no email sending, no fraud scoring inside the lock. Accept the
bid, commit, then publish an event.

In [ ]:
# Helper: reset an auction to a known state for each experiment

def reset_auction(auction_id, starting_max=1000.00):
    """Reset an auction's max bid so we can re-run experiments cleanly."""
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = NULL WHERE id = %s",
        (starting_max, auction_id)
    )
    # Clean up any test bids
    cur.execute("DELETE FROM bids WHERE auction_id = %s AND amount >= 5000", (auction_id,))
    conn.close()
    print(f"🔄 Auction {auction_id} reset to max_bid = ${starting_max:.2f}")

def show_auction(auction_id):
    """Display the current state of an auction."""
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT a.id, i.name, a.max_bid_amount, a.max_bid_user_id, a.status
           FROM auctions a JOIN items i ON a.item_id = i.id
           WHERE a.id = %s""",
        (auction_id,)
    )
    row = cur.fetchone()
    conn.close()
    if row:
        print(f"🏷️  Auction #{row[0]}: {row[1]}")
        print(f"   Current highest bid: ${row[2]:,.2f}")
        print(f"   Leading bidder: User {row[3] or 'None'}")
        print(f"   Status: {row[4]}")
    return row

# Let's see a sample auction
print("📋 Sample auction from our database:\n")
show_auction(1)

---
## 🐛 The Problem: Race Conditions

Imagine this scenario:

1. The current highest bid on a guitar is **$1,000**
2. **User A** wants to bid **$5,000** (a big jump!)
3. **User B** wants to bid **$1,500** at almost the same time

With a naive approach (read the max, check if your bid is higher, then write), this can happen:

```
Time    User A                          User B
─────   ──────────────────────────      ──────────────────────────
T1      Read max_bid → $1,000
T2                                      Read max_bid → $1,000
T3      $5,000 > $1,000? ✅ Accept
T4      Write max_bid = $5,000
T5                                      $1,500 > $1,000? ✅ Accept
T6                                      Write max_bid = $1,500  ← BUG!
```

User B's $1,500 bid **overwrites** User A's $5,000 bid! The auction now shows $1,500 as the highest bid. User A got robbed.

Let's reproduce this bug with real code.

In [ ]:
# ❌ BROKEN: Naive bid placement (read-then-write, no protection)

def place_bid_naive(auction_id, user_id, amount):
    """
    Place a bid WITHOUT any concurrency protection.
    This is how a beginner might write it — and it's broken.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    # Step 1: Read the current max bid
    cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
    current_max = float(cur.fetchone()[0])

    # Simulate network delay — this makes the race condition more likely
    time.sleep(0.1)

    # Step 2: Check if our bid is higher
    if amount > current_max:
        # Step 3: Write the new max bid
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
            (amount, user_id, auction_id)
        )
        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
            (auction_id, user_id, amount)
        )
        conn.commit()
        conn.close()
        return {"status": "accepted", "amount": amount, "user_id": user_id}
    else:
        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
            (auction_id, user_id, amount)
        )
        conn.commit()
        conn.close()
        return {"status": "rejected", "amount": amount, "user_id": user_id}

In [ ]:
# Let's trigger the race condition!
# We use auction 16 (Dyson vacuum, starts at $300, no bids yet)

AUCTION_ID = 16


def current_max(auction_id):
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
    value = float(cur.fetchone()[0])
    conn.close()
    return value


# User A's request arrives first; User B's arrives STAGGER seconds later,
# still inside A's 0.1 s "network delay" window. That is the ordinary case:
# not simultaneous to the microsecond, just overlapping. Staggering rather
# than racing makes the interleaving reproducible, so the demo teaches the
# same lesson on every run instead of flipping a coin.
STAGGER = 0.02


def run_two_bidders(bid_fn, rounds=3, quiet=False):
    """User A bids $5,000; User B bids $1,500 a moment later.

    Both read the same starting max of $1,000. Whatever the interleaving,
    the ONLY correct final state is max_bid_amount == 5000.
    Returns the final max of each round.
    """
    finals = []
    for rnd in range(1, rounds + 1):
        conn = get_db_connection()
        conn.autocommit = True
        cur = conn.cursor()
        cur.execute(
            "UPDATE auctions SET max_bid_amount = 1000.00, max_bid_user_id = NULL WHERE id = %s",
            (AUCTION_ID,))
        cur.execute("DELETE FROM bids WHERE auction_id = %s AND amount >= 1500", (AUCTION_ID,))
        conn.close()

        results = {}

        def bid_thread(name, user_id, amount):
            results[name] = bid_fn(AUCTION_ID, user_id, amount)

        thread_a = threading.Thread(target=bid_thread, args=("User A", 10, 5000.00))
        thread_b = threading.Thread(target=bid_thread, args=("User B", 20, 1500.00))
        thread_a.start()
        time.sleep(STAGGER)
        thread_b.start()
        thread_a.join()
        thread_b.join()

        final = current_max(AUCTION_ID)
        finals.append(final)
        if not quiet:
            outcome = "  ".join(
                f"{n}=${results[n]['amount']:,.0f}:{results[n]['status']}"
                for n in ("User A", "User B"))
            verdict = "✅ correct" if final == 5000.0 else "💥 LOST BID"
            print(f"  round {rnd}: final max=${final:>8,.2f}  {verdict}   {outcome}")
    return finals


print("❌ Naive read-then-write, 3 rounds:")
print("-" * 72)
naive_finals = run_two_bidders(place_bid_naive, rounds=3)

lost = [f for f in naive_finals if f != 5000.0]
print()
print(f"📋 Final auction state after the last round:")
show_auction(AUCTION_ID)
print()
assert lost, "expected at least one lost bid — the race did not reproduce"
print(f"⚠️  {len(lost)}/{len(naive_finals)} rounds ended with the wrong winner: the")
print(f"   $1,500 bid overwrote the $5,000 bid and the auction now shows "
      f"${lost[-1]:,.2f}.")
print("   Both bidders were told 'accepted'. User A is going to be very confused.")

---
## 🔒 Fix 1: Row-Level Locking (Pessimistic Approach)

The simplest fix is to **lock the auction row** while processing a bid. This forces concurrent bids to wait in line.

### How It Works
1. `BEGIN` a transaction
2. `SELECT ... FOR UPDATE` — this locks the auction row. Any other transaction trying to read the same row will **wait** until we're done.
3. Check if the new bid is higher than the current max
4. If yes, update the auction and insert the bid
5. `COMMIT` — this releases the lock

### Why Lock the Auction Row (Not All Bid Rows)?
An earlier approach in the source material locks **all bid rows** (`SELECT * FROM bids WHERE auction_id = ? FOR UPDATE`). This is bad because:
- As bids grow, you lock more and more rows
- `FOR UPDATE` only locks **existing** rows, not new inserts
- Performance degrades over time

Instead, we lock **one row** on the `auctions` table. This is fast and scales well.

In [ ]:
# ✅ FIXED: Bid placement with row-level locking (pessimistic concurrency)

def place_bid_with_lock(auction_id, user_id, amount):
    """
    Place a bid using SELECT ... FOR UPDATE to lock the auction row.
    This guarantees only one bid is processed at a time for this auction.
    """
    conn = get_db_connection()
    cur = conn.cursor()

    try:
        # Step 1: Lock the auction row and read the current max bid
        # FOR UPDATE tells Postgres: "Lock this row — nobody else can read or
        # modify it until I commit or rollback."
        cur.execute(
            "SELECT max_bid_amount FROM auctions WHERE id = %s FOR UPDATE",
            (auction_id,)
        )
        current_max = float(cur.fetchone()[0])

        # Simulate network delay — even with this, the lock keeps us safe
        time.sleep(0.1)

        # Step 2: Check and write
        if amount > current_max:
            cur.execute(
                "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
                (amount, user_id, auction_id)
            )
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "accepted", "amount": amount, "user_id": user_id}
        else:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount)
            )
            conn.commit()
            conn.close()
            return {"status": "rejected", "amount": amount, "user_id": user_id}
    except Exception as e:
        conn.rollback()
        conn.close()
        return {"status": "error", "amount": amount, "user_id": user_id, "error": str(e)}

In [ ]:
# Test: Same scenario, but now with locking. 5 rounds — a concurrency fix
# that works once has proven nothing.

print("✅ Row-level locking (SELECT ... FOR UPDATE), 5 rounds:")
print("-" * 72)
locked_finals = run_two_bidders(place_bid_with_lock, rounds=5)

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)
print()

assert all(f == 5000.0 for f in locked_finals), f"lost a bid: {locked_finals}"
print("🎉 5/5 rounds ended at $5,000. The $1,500 bid can never win.")
print()
print("   Here User A takes the lock first, so User B waits, re-reads $5,000")
print("   and is correctly rejected. Reverse the arrival order and BOTH bids")
print("   are accepted ($1,500 then $5,000) — which is also correct. The")
print("   invariant is 'the highest bid wins', not 'the loser is rejected'.")
print()
print("   What the lock costs: every bid on this auction is now serialised.")
print("   Two bidders take 2 x 100 ms instead of 100 ms in parallel — measure")
print("   it by comparing the wall-clock time of this cell against the naive one.")

### Tradeoffs of Row-Level Locking

| Pro | Con |
|-----|-----|
| Simple to implement | Bids are serialized — one at a time per auction |
| Strong consistency guaranteed | Waiting threads block (wasted resources) |
| Works with any SQL database | Lock contention increases with popularity |

For a hot auction with hundreds of bids per second, this approach can become a bottleneck. The next approach avoids locking entirely.

---
## ⏰ The Bug All Three Fixes Share: Bidding on a Closed Auction

Row locking fixed the lost update. It did **not** make `place_bid_with_lock`
correct, and neither will OCC or the Redis script below — because all three
answer only one question:

> *"Is this bid higher than the current max?"*

They never ask the other one:

> *"Is this auction still open?"*

`place_bid_with_lock` has no idea `auctions.status` or `auctions.end_date`
exist. Bid on an auction that ended last Tuesday and it will cheerfully accept
it and rewrite the winner.

This is the failure that actually costs money. A lost bid is embarrassing; a bid
accepted after close means you sold the item to the wrong person, and the real
winner has a screenshot.

In [ ]:
# ── Set up an auction that has already ENDED ────────────────────────────
def make_closed_auction(seconds_ago=60, status="ended", max_bid=750.00, winner=30):
    """Create a fresh auction whose end_date is in the past."""
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()
    cur.execute(
        "INSERT INTO items (seller_id, name, description) VALUES (1, %s, %s) RETURNING id",
        ("Closed Auction Test Item", "Used to demo bidding after close"))
    item_id = cur.fetchone()[0]
    cur.execute(
        """INSERT INTO auctions (item_id, seller_id, starting_price, max_bid_amount,
                                 max_bid_user_id, start_date, end_date, status)
           VALUES (%s, 1, 100.00, %s, %s,
                   NOW() - interval '7 days',
                   NOW() - (%s * interval '1 second'), %s)
           RETURNING id""",
        (item_id, max_bid, winner, seconds_ago, status))
    auction_id = cur.fetchone()[0]
    conn.close()
    return auction_id


def auction_state(auction_id):
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT status, max_bid_amount, max_bid_user_id,
                  EXTRACT(EPOCH FROM (end_date - NOW()))
           FROM auctions WHERE id = %s""", (auction_id,))
    row = cur.fetchone()
    conn.close()
    return {"status": row[0], "max_bid": float(row[1]),
            "winner": row[2], "seconds_left": float(row[3])}


closed_id = make_closed_auction(seconds_ago=60, status="ended")
before = auction_state(closed_id)
print(f"🔒 Auction #{closed_id}: status={before['status']}, "
      f"ended {abs(before['seconds_left']):.0f}s ago, "
      f"winner=User {before['winner']} at ${before['max_bid']:,.2f}")

# A latecomer bids on the closed auction using our "fixed" function.
late = place_bid_with_lock(closed_id, user_id=41, amount=9999.00)
after = auction_state(closed_id)

print(f"\n   Late bid from User 41 for $9,999 → {late['status']}")
print(f"   Auction is now: status={after['status']}, "
      f"winner=User {after['winner']} at ${after['max_bid']:,.2f}")

assert late["status"] == "accepted" and after["winner"] == 41, \
    "expected the unguarded function to accept a bid on a closed auction"
print()
print("💥 The auction was ALREADY ENDED and we changed the winner. Row locking")
print("   protected us from a concurrent bidder and did nothing about the clock.")

# The same hole with an auction that is still 'active' but past its end_date —
# i.e. any auction in the window between expiring and the closer job noticing.
expired_id = make_closed_auction(seconds_ago=5, status="active", winner=31)
place_bid_with_lock(expired_id, user_id=42, amount=8888.00)
st = auction_state(expired_id)
assert st["winner"] == 42
print()
print(f"   Same story for auction #{expired_id}: status is still 'active' but it")
print(f"   expired {abs(st['seconds_left']):.0f}s ago and the closer job hasn't run yet.")
print("   That gap is not a corner case — it is however long your closer's")
print("   polling interval is, on every auction, every time.")

### The Fix: Validate the Clock Inside the Same Lock

Two rules, and the second one is the one people get wrong:

1. **Check `status` and `end_date`**, not just the amount.
2. **Do it inside the same transaction that holds the row lock**, and let
   *Postgres* evaluate `NOW()`.

Rule 2 matters because a check outside the lock is another
time-of-check-to-time-of-use race, and because the *application server's* clock
is not the auction's clock. Two API servers with 200 ms of clock skew will
disagree about whether an auction is over — and in an auction, 200 ms at the
end is exactly the moment everyone is bidding. Comparing `datetime.now()` in
Python against a timestamp from the database is a bug waiting for a busy day.

There is only one clock that matters, and it's the database's.

In [ ]:
def place_bid_guarded(auction_id, user_id, amount):
    """Row lock + amount check + CLOSE check, all in one transaction.

    Everything time-related is evaluated by Postgres (NOW()), so the API
    server's clock is irrelevant.
    """
    conn = get_db_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            """SELECT max_bid_amount, status, end_date <= NOW() AS expired
               FROM auctions WHERE id = %s FOR UPDATE""",
            (auction_id,))
        row = cur.fetchone()
        if row is None:
            conn.rollback()
            return {"status": "rejected", "reason": "no such auction"}

        current_max, status, expired = float(row[0]), row[1], row[2]

        # Reject BEFORE touching anything. Note we still record the attempt —
        # a rejected bid is evidence in a dispute, so never drop it silently.
        if status != "active":
            reason = f"auction is {status}"
        elif expired:
            reason = "auction has ended"
        elif amount <= current_max:
            reason = f"${amount:,.2f} does not beat ${current_max:,.2f}"
        else:
            reason = None

        if reason:
            cur.execute(
                "INSERT INTO bids (auction_id, user_id, amount, status)"
                " VALUES (%s, %s, %s, 'rejected')",
                (auction_id, user_id, amount))
            conn.commit()
            return {"status": "rejected", "reason": reason}

        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s,"
            " updated_at = NOW() WHERE id = %s",
            (amount, user_id, auction_id))
        cur.execute(
            "INSERT INTO bids (auction_id, user_id, amount, status)"
            " VALUES (%s, %s, %s, 'accepted')",
            (auction_id, user_id, amount))
        conn.commit()
        return {"status": "accepted", "amount": amount, "user_id": user_id}
    except Exception as e:
        conn.rollback()
        return {"status": "error", "reason": str(e)}
    finally:
        conn.close()


print("Guarded bid function vs. the three ways an auction can be un-biddable:")
print("-" * 72)
for label, aid, uid, amt in [
    ("ended auction",            make_closed_auction(60, "ended", 750.00, 30), 41, 9999.00),
    ("expired but still active", make_closed_auction(5, "active", 750.00, 31), 42, 8888.00),
    ("bid too low",              make_closed_auction(-3600, "active", 750.00, 32), 43, 500.00),
]:
    winner_before = auction_state(aid)["winner"]
    res = place_bid_guarded(aid, uid, amt)
    st = auction_state(aid)
    print(f"  {label:<26} → {res['status']:<8} ({res.get('reason', '')})")
    assert res["status"] == "rejected", res
    assert st["winner"] == winner_before, "a rejected bid must not change the winner"

# …and it still lets a legitimate bid through.
open_id = make_closed_auction(-3600, "active", 750.00, 33)   # ends in an hour
ok = place_bid_guarded(open_id, user_id=44, amount=1200.00)
assert ok["status"] == "accepted" and auction_state(open_id)["winner"] == 44
print(f"  {'open auction, higher bid':<26} → accepted (winner is now User 44)")

print()
print("✅ Every closed-auction bid rejected, the winner never changed, and a")
print("   legitimate bid still goes through.")
print()
print("What this costs: the reject path still writes a row to `bids`. That is")
print("deliberate — 'I bid $9,999 and got nothing' is a support ticket you want")
print("to be able to answer with data. It also means a bot hammering a closed")
print("auction generates write load, so rate-limit at the edge, not here.")

### One More Race: The Bidder and the Closer

The guard fixes the *sequential* case. There's still a concurrent one, and it's
the interesting one: at the exact instant an auction expires, the **closer job**
is deciding the winner while a **bidder** is trying to get one last bid in.

If those two run without contending on the same row, you can end up with a bid
accepted *after* the winner was recorded — the auction says User 30 won, and the
bid table says User 99 bid higher, still marked accepted.

Both sides take `FOR UPDATE` on the same auction row, so they serialise: either
the bid lands before the close (and the closer sees it) or the close lands first
(and the guard rejects the bid). Let's fire them simultaneously and check the
invariant that actually matters:

> **The recorded winner must equal the highest accepted bid.**

In [ ]:
def close_auction_guarded(auction_id):
    """Closer job for one auction. Takes the same row lock the bidder does,
    and re-checks expiry inside the lock (an anti-snipe extension may have
    moved end_date since we selected the candidate)."""
    conn = get_db_connection()
    cur = conn.cursor()
    try:
        cur.execute(
            """SELECT status, end_date <= NOW() FROM auctions
               WHERE id = %s FOR UPDATE""", (auction_id,))
        status, expired = cur.fetchone()
        if status != "active" or not expired:
            conn.rollback()
            return {"closed": False, "reason": f"status={status} expired={expired}"}
        cur.execute(
            "UPDATE auctions SET status = 'ended', updated_at = NOW()"
            " WHERE id = %s RETURNING max_bid_amount, max_bid_user_id",
            (auction_id,))
        amount, winner = cur.fetchone()
        conn.commit()
        return {"closed": True, "winner": winner, "amount": float(amount)}
    finally:
        conn.close()


def highest_accepted_bid(auction_id):
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute(
        """SELECT user_id, amount FROM bids
           WHERE auction_id = %s AND status = 'accepted'
           ORDER BY amount DESC LIMIT 1""", (auction_id,))
    row = cur.fetchone()
    conn.close()
    return (row[0], float(row[1])) if row else (None, None)


print("🏁 Bidder vs closer, fired simultaneously, 6 rounds")
print("=" * 78)
# Sweep the expiry across the moment of contention: negative = still open by
# that many seconds, positive = already expired. Real traffic hits every point
# on this line; we walk it deliberately so both branches actually show up.
tally = {"bid won": 0, "closer won": 0}
for rnd, offset in enumerate([-0.10, -0.05, -0.02, 0.0, 0.02, 0.05], start=1):
    aid = make_closed_auction(seconds_ago=offset, status="active",
                              max_bid=750.00, winner=30)

    outcome = {}
    gate = threading.Barrier(2)

    def bidder():
        gate.wait()
        outcome["bid"] = place_bid_guarded(aid, user_id=41, amount=9999.00)

    def closer():
        gate.wait()
        outcome["close"] = close_auction_guarded(aid)

    threads = [threading.Thread(target=bidder), threading.Thread(target=closer)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    st = auction_state(aid)
    top_user, top_amount = highest_accepted_bid(aid)
    tally["bid won" if outcome["bid"]["status"] == "accepted" else "closer won"] += 1
    print(f"  round {rnd}: ends in {-offset:+.2f}s  bid={outcome['bid']['status']:<8} "
          f"closed={'yes' if outcome['close']['closed'] else 'no ':<3}  "
          f"winner=User {st['winner']} @ ${st['max_bid']:,.2f}  "
          f"top accepted bid={top_user}")

    # THE invariant: whoever the auction says won must be the highest
    # accepted bidder. Anything else means we sold to the wrong person.
    if top_user is not None:
        assert st["winner"] == top_user, (
            f"winner mismatch: auction says {st['winner']}, "
            f"highest accepted bid is {top_user}")
    assert st["max_bid"] >= 750.00

print()
print(f"✅ 6/6 rounds: the recorded winner is always the highest accepted bidder.")
print(f"   Outcomes: bidder got in first {tally['bid won']}x, "
      f"closer got there first {tally['closer won']}x.")
print()
print("   BOTH are correct, and which one you get is genuinely a race:")
print("     • bidder wins the lock → bid accepted; the closer then finds the")
print("       auction has NOT expired yet and correctly leaves it open. The")
print("       next closer pass ends it with User 41 as the winner.")
print("     • closer wins the lock → auction ends first, and the guard rejects")
print("       the bid with 'auction has ended'")
print()
print("   What is NOT legitimate, and what the naive version produced, is an")
print("   accepted bid that the recorded winner doesn't reflect. 'Last write")
print("   wins' is not a tiebreak rule you can defend to a customer.")

# Tidy up the throwaway auctions and items this section created.
conn = get_db_connection()
conn.autocommit = True
cur = conn.cursor()
cur.execute("""DELETE FROM bids WHERE auction_id IN (
                   SELECT a.id FROM auctions a JOIN items i ON i.id = a.item_id
                   WHERE i.name = 'Closed Auction Test Item')""")
cur.execute("""DELETE FROM auctions WHERE item_id IN (
                   SELECT id FROM items WHERE name = 'Closed Auction Test Item')""")
cur.execute("DELETE FROM items WHERE name = 'Closed Auction Test Item'")
conn.close()
print("\n🧹 Cleaned up the throwaway auctions from this section.")

---
## ⚡ Fix 2: Optimistic Concurrency Control (OCC)

**Key insight**: Bid conflicts are actually **rare**. Most bids don't happen at the exact same millisecond. So instead of locking (which is expensive), we can be **optimistic**:

1. Read the current max bid (no lock!)
2. Try to update the auction, **but only if the max bid hasn't changed** since we read it
3. If someone else updated it first, our update touches 0 rows — we detect this and **retry**

The magic is in this SQL:
```sql
UPDATE auctions
SET max_bid_amount = $new_bid
WHERE id = $auction_id AND max_bid_amount = $original_max
```

The `AND max_bid_amount = $original_max` is the **version check**. If someone else changed it between our read and write, this `WHERE` clause won't match, and zero rows get updated.

In [ ]:
# ✅ FIXED: Bid placement with Optimistic Concurrency Control

def place_bid_occ(auction_id, user_id, amount, max_retries=5):
    """
    Place a bid using Optimistic Concurrency Control.
    No locks! Instead, we detect conflicts and retry.
    """
    for attempt in range(max_retries):
        conn = get_db_connection()
        conn.autocommit = False
        cur = conn.cursor()

        try:
            # Step 1: Read the current max bid (no lock)
            cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (auction_id,))
            current_max = float(cur.fetchone()[0])

            # Is our bid even high enough?
            if amount <= current_max:
                cur.execute(
                    "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'rejected')",
                    (auction_id, user_id, amount)
                )
                conn.commit()
                conn.close()
                return {"status": "rejected", "amount": amount, "user_id": user_id, "attempts": attempt + 1}

            # Step 2: Try to update — but ONLY if max_bid hasn't changed
            cur.execute(
                """UPDATE auctions
                   SET max_bid_amount = %s, max_bid_user_id = %s
                   WHERE id = %s AND max_bid_amount = %s""",
                (amount, user_id, auction_id, current_max)
            )

            # Check if the update actually changed a row
            if cur.rowcount == 1:
                # Success! Nobody changed the max bid between our read and write.
                cur.execute(
                    "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, 'accepted')",
                    (auction_id, user_id, amount)
                )
                conn.commit()
                conn.close()
                return {"status": "accepted", "amount": amount, "user_id": user_id, "attempts": attempt + 1}
            else:
                # Conflict! Someone else updated the max bid. Rollback and retry.
                conn.rollback()
                conn.close()
                print(f"   🔄 User {user_id}: Conflict on attempt {attempt + 1}, retrying...")
                continue

        except Exception as e:
            conn.rollback()
            conn.close()
            return {"status": "error", "amount": amount, "user_id": user_id, "error": str(e)}

    return {"status": "failed_after_retries", "amount": amount, "user_id": user_id}

In [ ]:
# Test: OCC with concurrent bids

reset_auction(AUCTION_ID, starting_max=1000.00)
print()

results = {}

def bid_thread_occ(name, auction_id, user_id, amount):
    result = place_bid_occ(auction_id, user_id, amount)
    results[name] = result

thread_a = threading.Thread(target=bid_thread_occ, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread_occ, args=("User B", AUCTION_ID, 20, 1500.00))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results with OCC:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    attempts = result.get("attempts", "?")
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']} (attempts: {attempts})")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("🎉 Correct result achieved without any locks!")
print("   If there was a conflict, the losing transaction simply retried.")

In [ ]:
# Stress test: 10 concurrent bidders, each bidding a different amount

reset_auction(AUCTION_ID, starting_max=1000.00)
print()
print("🏁 Stress test: 10 users bidding at the same time...\n")

stress_results = {}

def stress_bid(user_id, amount):
    result = place_bid_occ(AUCTION_ID, user_id, amount)
    stress_results[user_id] = result

threads = []
bids = [
    (1, 5100), (2, 5200), (3, 5300), (4, 5400), (5, 5500),
    (6, 5600), (7, 5700), (8, 5800), (9, 5900), (10, 6000),
]

for user_id, amount in bids:
    t = threading.Thread(target=stress_bid, args=(user_id, amount))
    threads.append(t)

# Start all 10 threads at once
for t in threads:
    t.start()
for t in threads:
    t.join()

print("\n📊 Stress test results:")
for user_id in sorted(stress_results.keys()):
    r = stress_results[user_id]
    emoji = "✅" if r["status"] == "accepted" else "❌"
    print(f"   {emoji} User {user_id:>2} bid ${r['amount']:>8,.2f} → {r['status']:<10} (attempts: {r.get('attempts', '?')})")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

print()
print("💡 Notice how some users needed multiple attempts (retries) due to conflicts.")
print("   But the final result is always correct — the highest bid wins.")

### Tradeoffs of OCC

| Pro | Con |
|-----|-----|
| No locks — reads never block | Retries needed on conflict |
| Higher throughput under low contention | Under very high contention, many retries |
| Simple to implement | Wasted work on failed attempts |

**OCC is ideal for auctions** because bid conflicts are relatively rare. Most of the time, bids arrive seconds apart, not at the exact same moment.

---
## 🚀 Fix 3: Redis Atomic Compare-and-Set (Lua Script)

What if we move the "current max bid" into **Redis** instead of reading it from Postgres each time?

Redis is single-threaded, so operations run one at a time. By using a **Lua script**, we can read the current max, compare, and set the new max as **one atomic operation** — no locks, no retries.

### How It Works
1. Store the current max bid for each auction in Redis: `auction:{id}:max_bid`
2. When a bid comes in, run a Lua script that:
   - Reads the current max from Redis
   - Compares the new bid against it
   - If higher, updates Redis and returns `1` (accepted)
   - If not, returns `0` (rejected)
3. If accepted, write the bid to Postgres (permanent storage)

### Why Lua?
Redis `MULTI/EXEC` (transactions) can't read a value and conditionally write based on it. Lua scripts run atomically inside Redis — they can do read-modify-write in one step.

In [ ]:
# ✅ FIXED: Bid placement with Redis Lua script (atomic compare-and-set)

# This Lua script runs INSIDE Redis — it's atomic (no other command can run
# between the GET and SET). Redis is single-threaded, so this is safe.
COMPARE_AND_SET_LUA = """
local current_max = tonumber(redis.call('GET', KEYS[1]) or '0')
local proposed_bid = tonumber(ARGV[1])
local user_id = ARGV[2]

if proposed_bid > current_max then
    redis.call('SET', KEYS[1], proposed_bid)
    redis.call('SET', KEYS[1] .. ':user', user_id)
    return 1
else
    return 0
end
"""

def place_bid_redis(auction_id, user_id, amount):
    """
    Place a bid using Redis for the fast atomic check,
    then persist to Postgres for durability.
    """
    r = get_redis_client()
    cache_key = f"auction:{auction_id}:max_bid"

    # Step 1: Atomic compare-and-set in Redis
    accepted = r.eval(COMPARE_AND_SET_LUA, 1, cache_key, str(amount), str(user_id))

    # Step 2: Write to Postgres for permanent storage
    conn = get_db_connection()
    conn.autocommit = True
    cur = conn.cursor()

    status = "accepted" if accepted == 1 else "rejected"
    cur.execute(
        "INSERT INTO bids (auction_id, user_id, amount, status) VALUES (%s, %s, %s, %s)",
        (auction_id, user_id, amount, status)
    )

    if accepted == 1:
        cur.execute(
            "UPDATE auctions SET max_bid_amount = %s, max_bid_user_id = %s WHERE id = %s",
            (amount, user_id, auction_id)
        )

    conn.close()
    return {"status": status, "amount": amount, "user_id": user_id}

In [ ]:
# Test: Redis atomic approach with concurrent bids

reset_auction(AUCTION_ID, starting_max=1000.00)

# Seed the Redis cache with the current max bid
r = get_redis_client()
r.set(f"auction:{AUCTION_ID}:max_bid", "1000.00")
print(f"🔄 Redis seeded: auction:{AUCTION_ID}:max_bid = 1000.00\n")

results = {}

def bid_thread_redis(name, auction_id, user_id, amount):
    result = place_bid_redis(auction_id, user_id, amount)
    results[name] = result

thread_a = threading.Thread(target=bid_thread_redis, args=("User A", AUCTION_ID, 10, 5000.00))
thread_b = threading.Thread(target=bid_thread_redis, args=("User B", AUCTION_ID, 20, 1500.00))

thread_a.start()
thread_b.start()
thread_a.join()
thread_b.join()

print("📊 Results with Redis Lua:")
for name, result in results.items():
    emoji = "✅" if result["status"] == "accepted" else "❌"
    print(f"   {emoji} {name} bid ${result['amount']:,.2f} → {result['status']}")

print()
print("📋 Final auction state:")
show_auction(AUCTION_ID)

# Show what Redis has
print()
print(f"📦 Redis state:")
print(f"   auction:{AUCTION_ID}:max_bid = {r.get(f'auction:{AUCTION_ID}:max_bid')}")
print(f"   auction:{AUCTION_ID}:max_bid:user = {r.get(f'auction:{AUCTION_ID}:max_bid:user')}")

# Cleanup Redis keys
r.delete(f"auction:{AUCTION_ID}:max_bid", f"auction:{AUCTION_ID}:max_bid:user")

### Tradeoffs of Redis Lua

| Pro | Con |
|-----|-----|
| Extremely fast (~1ms) | Two systems to keep in sync (Redis + Postgres) |
| No locks, no retries | If Redis and Postgres disagree, which is correct? |
| Scales independently of DB | Redis is in-memory — data can be lost on crash |

### The Consistency Challenge

The big question: **what happens if Redis says "accepted" but the Postgres write fails?**

Options:
1. **Accept Redis as source of truth** during the auction, write to Postgres async
2. **Write to Postgres first**, update Redis if DB succeeds, invalidate cache if it fails
3. **Use Redis only as a fast check**, then do the real write with OCC in Postgres

There's no perfect answer — this is why distributed consistency is one of the hardest problems in system design!

---
## 📊 Comparing All Three Approaches

Let's benchmark all three with a burst of 20 concurrent bids.

In [ ]:
import statistics

def benchmark(bid_fn, label, num_bids=20, setup_redis=False):
    """Run concurrent bids and measure total time and correctness."""
    reset_auction(AUCTION_ID, starting_max=1000.00)

    if setup_redis:
        r = get_redis_client()
        r.set(f"auction:{AUCTION_ID}:max_bid", "1000.00")

    results = {}
    def run_bid(idx):
        amount = 1100 + (idx * 100)
        start = time.time()
        result = bid_fn(AUCTION_ID, idx + 1, amount)
        elapsed = (time.time() - start) * 1000
        result["latency_ms"] = elapsed
        results[idx] = result

    threads = [threading.Thread(target=run_bid, args=(i,)) for i in range(num_bids)]

    start_time = time.time()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    total_time = (time.time() - start_time) * 1000

    # Check correctness — the highest amount should be the max bid
    expected_max = 1100 + (num_bids - 1) * 100
    conn = get_db_connection()
    cur = conn.cursor()
    cur.execute("SELECT max_bid_amount FROM auctions WHERE id = %s", (AUCTION_ID,))
    actual_max = float(cur.fetchone()[0])
    conn.close()

    accepted_count = sum(1 for r in results.values() if r["status"] == "accepted")
    latencies = [r["latency_ms"] for r in results.values()]

    if setup_redis:
        r = get_redis_client()
        r.delete(f"auction:{AUCTION_ID}:max_bid", f"auction:{AUCTION_ID}:max_bid:user")

    print(f"\n{'='*50}")
    print(f"📊 {label}")
    print(f"{'='*50}")
    correct = "✅ CORRECT" if actual_max == expected_max else f"❌ WRONG (expected {expected_max}, got {actual_max})"
    print(f"   Correctness:     {correct}")
    print(f"   Total time:      {total_time:.0f} ms")
    print(f"   Avg latency:     {statistics.mean(latencies):.0f} ms")
    print(f"   Bids accepted:   {accepted_count}/{num_bids}")

# Run all three benchmarks
benchmark(place_bid_naive,     "Naive (BROKEN — no protection)")
benchmark(place_bid_with_lock, "Row-Level Locking (Pessimistic)")
benchmark(place_bid_occ,       "Optimistic Concurrency Control")
benchmark(place_bid_redis,     "Redis Lua (Atomic Compare-and-Set)", setup_redis=True)

print("\n" + "="*50)
print("💡 Key takeaways:")
print("   - Naive is fast but WRONG — race conditions corrupt data")
print("   - Locking is correct but slow — bids are serialized")
print("   - OCC is correct and fast — retries handle rare conflicts")
print("   - Redis Lua is the fastest — but adds cross-system complexity")

---
## 🧠 Summary

| Approach | Consistency | Performance | Complexity | Best For |
|----------|------------|-------------|------------|----------|
| Naive (broken) | ❌ None | ⚡ Fast | Low | Nothing — it's broken |
| Row Locking | ✅ Strong | 🐢 Slow under contention | Low | Simple apps, low traffic |
| OCC | ✅ Strong | ⚡ Fast (retries on conflict) | Medium | Most auction systems |
| Redis Lua | ✅ Atomic in Redis | ⚡⚡ Very fast | High | High-throughput systems |

### In a Real System
- **OCC on the auction row** is the recommended approach for most cases
- **Redis Lua** is used when you need extreme throughput and accept the complexity of keeping Redis and Postgres in sync
- A **message queue** (Kafka) in front of the bid service adds durability — bids are never lost even if the service crashes

### What's Next
- **Notebook 2**: Auction lifecycle — creating, ending, and managing auction state
- **Notebook 3**: Real-time notifications — pushing bid updates to all watchers instantly